# 03. Advanced Trie: Word Search II, Stream of Characters, Maximum XOR

Tries combined with **backtracking**, **reversed words**, and **bits**.

**Learning order:** WORD SEARCH II → STREAM OF CHARACTERS → BINARY TRIE FOR MAX XOR → INTERVIEW THINKING

## 1. Word Search II

Find **all words** from a list that exist in a grid (path through adjacent cells, no cell reused). Running Word Search (backtracking) once per word is slow. Instead:

1. Insert **all words into a trie**.
2. DFS from every cell, following the trie **at the same time**. If the current letter has no trie child, stop (**pruning**).
3. When a node holds a word, add it and **clear it** so it is not added twice.
4. Optional: remove empty trie branches after exploring to speed up later searches.

In [1]:
def find_words(board, words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w                                # store the whole word at its end
    rows, cols = len(board), len(board[0])
    found = []

    def dfs(r, c, parent):
        ch = board[r][c]
        node = parent[ch]
        if "$" in node:
            found.append(node.pop("$"))              # take the word once
        board[r][c] = "#"                            # mark visited
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and board[nr][nc] in node:
                dfs(nr, nc, node)
        board[r][c] = ch                             # restore
        if not node:                                 # prune exhausted branch
            del parent[ch]

    for r in range(rows):
        for c in range(cols):
            if board[r][c] in root:
                dfs(r, c, root)
    return found

grid = [["o", "a", "a", "n"], ["e", "t", "a", "e"], ["i", "h", "k", "r"], ["i", "f", "l", "v"]]
print(sorted(find_words(grid, ["oath", "pea", "eat", "rain"])))   # ['eat', 'oath']

['eat', 'oath']


## 2. Stream of Characters

Letters arrive one at a time. After each letter, does **some word end exactly at the latest letter** (i.e. some word is a suffix of the stream)?

**Trick:** insert every word **reversed** into a trie. Keep the stream, and on each query walk the trie **backwards through the stream** (newest letter first). If you hit `is_end`, a word matches.

In [2]:
class StreamChecker:
    def __init__(self, words):
        self.root = {}
        self.max_len = 0
        for w in words:
            node = self.root
            for ch in reversed(w):
                node = node.setdefault(ch, {})
            node["$"] = True
            self.max_len = max(self.max_len, len(w))
        self.stream = []

    def query(self, letter):
        self.stream.append(letter)
        node = self.root
        for ch in reversed(self.stream[-self.max_len:]):
            if ch not in node:
                return False
            node = node[ch]
            if "$" in node:
                return True
        return False

sc = StreamChecker(["cd", "f", "kl"])
print([sc.query(c) for c in "abcdefghijkl"])   # [False, False, False, True, False, True, False, False, False, False, False, True]

[False, False, False, True, False, True, False, False, False, False, False, True]


## 3. Maximum XOR of Two Numbers (binary trie)

A number is a string of **bits**. Build a trie over the 32 bits (from the most significant). To maximize `x ^ y`, for each `x` walk the trie and at each bit **prefer the opposite bit** (it gives a `1` in the XOR). Higher bits matter more, so greedy works.

O(32 · n) instead of O(n²).

In [3]:
def find_maximum_xor(nums):
    root = {}
    for x in nums:
        node = root
        for i in range(31, -1, -1):
            bit = (x >> i) & 1
            node = node.setdefault(bit, {})
    best = 0
    for x in nums:
        node, cur = root, 0
        for i in range(31, -1, -1):
            bit = (x >> i) & 1
            want = 1 - bit
            if want in node:
                cur |= 1 << i
                node = node[want]
            else:
                node = node[bit]
        best = max(best, cur)
    return best

print(find_maximum_xor([3, 10, 5, 25, 2, 8]))   # 28  (5 ^ 25)

28


## 4. Interview thinking

**Signals:** "find all words in a grid", "word is a suffix of a stream", "maximum XOR", "prefix queries on numbers".

**Say:** "I put the words in a trie so one DFS can check many words at once and stop as soon as no word has this prefix."

**Common mistakes:** running Word Search once per word (TLE), adding the same word twice (forgot to clear it), not restoring the board cell, forgetting to reverse the words in Stream of Characters, using strings instead of bits for max XOR.

## Quick Revision

- Word Search II: trie of all words + one grid DFS, store the word at its end node, remove it when found, prune empty branches.
- Suffix of a stream → trie of **reversed** words, walk the stream backwards.
- Max XOR: binary trie, prefer the opposite bit at each level.